In [1]:
# Get raw data
with open('input/22.txt', 'r') as f:
    rawinput = f.read().strip()

In [2]:
# Part 1
blocks = [[min(j[k],j[k+3]) for k in [0,1,2]]
          +[max(j[k],j[k+3])+1 for k in [0,1,2]]
          for i in rawinput.split('\n')
          for j in [[int(u) 
                     for t in i.split('~') 
                     for u in t.split(',')]]]
xmax,ymax,zmax = [max(i[j]) for i in [[*zip(*blocks)]] for j in [3,4,5]]
blocks.insert(0,[0,0,0,xmax,ymax,1])
xyarea = [(i[3]-i[0])*(i[4]-i[1]) for i in blocks]

columns = {(x,y): [-1]*(zmax-1)
           for x in range(xmax)
           for y in range(ymax)}
for i,j in enumerate(blocks):
    for x in range(j[0],j[3]):
        for y in range(j[1],j[4]):
            columns[(x,y)][j[2]:j[5]] = [i]*(j[5]-j[2])
zht = {(x,y): 1
       for x in range(xmax)
       for y in range(ymax)}

getfirst = lambda x: None if {*x}=={-1} else x[0] if x[0]!=-1 else getfirst(x[1:])

while (cb:=[[blocks[b][2],b]
            for firstbycol in [[z 
                                for k in columns
                                for z in [getfirst(columns[k][zht[k]:])]
                                if z]]
            for b in {*firstbycol}
            if firstbycol.count(b)==xyarea[b]]):
    bht, block = sorted(cb)[0]
    zdrop = bht - max([zht[(x,y)]
                       for i in [blocks[block]]
                       for x in range(i[0],i[3])
                       for y in range(i[1],i[4])])
    b = blocks[block]
    for x in range(b[0],b[3]):
        for y in range(b[1],b[4]):
            columns[(x,y)][b[2]:b[5]] = [-1]*(z:=(b[5]-b[2]))
            columns[(x,y)][b[2]-zdrop:b[5]-zdrop] = [block]*z
            zht[(x,y)] = b[5]-zdrop
    b[2]-=zdrop
    b[5]-=zdrop

safeblocks = {*range(1,len(blocks))}
for b in range(1,len(blocks)):
    ro = [i
          for i in sorted({columns[(x,y)][blocks[b][2]-1]
                           for x in range(blocks[b][0],blocks[b][3])
                           for y in range(blocks[b][1],blocks[b][4])})
          if i!=-1]
    if len(ro)==1:
        safeblocks -= {*ro}
len(safeblocks)

428

In [3]:
# Part 2
rodict = {b: {i
              for i in sorted({columns[(x,y)][blocks[b][2]-1]
                               for x in range(blocks[b][0],blocks[b][3])
                               for y in range(blocks[b][1],blocks[b][4])})
              if i!=-1}
          for b in range(1,len(blocks))}
suppdict = dict()
for k,v in rodict.items():
    for i in v:
        suppdict[i] = suppdict.get(i,[])+[k]
        
evolve = lambda x: (x | {j
                         for i in x 
                         for j in suppdict.get(i,[])
                         if j not in x
                         and all([k in x for k in rodict[j]])})

def get_n_falling(block):
    falling = {block}
    while falling != (falling:=evolve(falling)):  pass
    return len(falling-{block})

sum([get_n_falling(i) for i in range(1,len(blocks))])

35654